In [ ]:
# ==============================================================================
# External Cross-Generator Test — English track
# MODIFIED: the 'Proposed Method' slot now loads the NEW head from
# DeBERTaV3_SupCon_MLP_AttnPool (SupCon encoder + MLPHead sentence head +
# AttnPool doc head whose classifier is the shared MLPHead class). Changes vs
# the previous version: checkpoint folder/file names, state_dict keys
# (attn_doc_head_state_dict), MLPHead classifier (classifier.net.*),
# config keys (config_*.json, s3_best_val_f1), and sentence pipeline
# (normalize whole doc -> split -> >=8 words) exactly as in training.
# (see chat, 2026-10-20): REBUILD of
# external_cross_generator_test_six_doclevel_plus_dualsvm.py, swapping the
# seventh slot: Dual-SVM (Doc mean-pool) is REMOVED and replaced with
# "Attention-Pooling (Doc)" — the OLD method built on the SAME frozen
# SupCon (contrastive-learning) encoder as Dual-SVM used
# (DeBERTaV3_SupCon_SVM/encoder.pt), from
# deberta_v3_supcon_attnpool_dochead_english.py's trained head
# (DeBERTaV3_SupCon_AttnPool_DocHead/attnpool_doc_head.pt). This is NOT the
# newer "DeBERTa-v3 FT + Attn-Pool (Doc)" method added to the RAID script
# (that one's encoder was itself fine-tuned with Cross-Entropy in Stage 2,
# not just contrastively) — this is specifically the ORIGINAL
# Attention-Pooling head sitting on top of the purely contrastive
# (SupCon-only), externally frozen encoder.
#   - The six standard classifiers (BERT, RoBERTa, ELECTRA, DeBERTa-v3,
#     XLNet, DistilBERT) are UNCHANGED — their DOCUMENT-LEVEL fine-tuned
#     checkpoints, each document fed as ONE sequence (MAX_LENGTH=512).
#   - The "Proposed Method" slot is now Attention-Pooling (Doc), from
#     deberta_v3_supcon_attnpool_dochead_english.py: sentence embeddings
#     from the frozen SupCon encoder, combined via a learned masked-softmax
#     attention-weighted sum (early fusion) instead of Dual-SVM's plain
#     mean-pool + SVM.
# DOCUMENT-LEVEL ONLY, for ALL SEVEN methods — there is no sentence-level
# table, confusion matrix, ROC panel or breakdown anywhere in this script.
# ------------------------------------------------------------------------------
# Single MERGED input file ("Five.xlsx": text, label, model_used, ...).
# Human-written rows appear ONCE (not duplicated per generator):
#   - "Human" row  = specificity — correct rate on label==0 rows
#   - one row PER unseen AI generator (from model_used) = recall — correct
#     rate on that generator's label==1 rows only
# Same Q1-journal plot theme (sequential blue heatmap, dynamic per-cell text
# color, 300 DPI) as the rest of this project's cross-generator/OOD tests.
# ==============================================================================
# قبل التشغيل بخلية منفصلة:
#   !pip install -q -U transformers sentencepiece

import os, re, shutil, unicodedata, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns

from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    confusion_matrix, roc_curve, auc, matthews_corrcoef,
)
from tqdm.auto import tqdm

from google.colab import drive
drive.mount("/content/drive", force_remount=True)

pd.set_option("display.width", 130)
pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ==============================================================================
# 0. Q1-JOURNAL PUBLICATION PLOT THEME — SAME helpers as the reference
# template (sequential blue heatmap, panel letters, white-halo bar labels,
# 300 DPI), copied verbatim.
# ==============================================================================
ATTN_KEY = "Proposed Method"   # renamed from "Attention-Pooling (Doc)" (see chat, 2026-10-23)

METHOD_ORDER = ["BERT", "RoBERTa", "ELECTRA", "DeBERTa-v3", "XLNet", "DistilBERT", ATTN_KEY]
METHOD_COLORS = ["#2E5EAA", "#22A6B3", "#4C72B0", "#C0392B", "#8E44AD", "#7F8C8D", "#D35400"]
CM_CMAPS = ["Blues", "GnBu", "PuBu", "Reds", "Purples", "Greys", "OrRd"]

DISPLAY_NAMES = {}   # no renaming needed — real method names shown everywhere
def disp(m):
    return DISPLAY_NAMES.get(m, m)

BLUE_ONLY_CMAP = LinearSegmentedColormap.from_list(
    "light_to_dark_blue",
    ["#F5F9FD", "#D6E7F7", "#B0D0EF", "#82B4E3", "#4F92D4", "#2E72B8", "#1A5596"],
    N=256,
)
GRID_COLOR = "#DDE3EC"
AXIS_COLOR = "#2B2B2B"

def heatmap_dynamic_text(ax, data, vmin, vmax, cmap, fmt="{:.3f}", fontsize=13):
    span = max(vmax - vmin, 1e-9)
    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            val = data.iloc[i, j] if hasattr(data, "iloc") else data[i][j]
            norm = (float(val) - vmin) / span
            r, g, b, _ = cmap(np.clip(norm, 0, 1))
            luminance = 0.299 * r + 0.587 * g + 0.114 * b
            text_color = "white" if luminance < 0.55 else "#1A1A1A"
            label = fmt.format(val) if not isinstance(val, str) else val
            ax.text(j + 0.5, i + 0.5, label, ha="center", va="center",
                    fontsize=fontsize, fontweight="bold", color=text_color)

plt.rcParams.update({
    "figure.facecolor": "white", "axes.facecolor": "white",
    "axes.edgecolor": AXIS_COLOR, "axes.linewidth": 1.1,
    "axes.grid": False, "axes.axisbelow": True,
    "grid.color": GRID_COLOR, "grid.linewidth": 0.7, "grid.alpha": 0.9,
    "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 11.5, "font.family": "DejaVu Serif",
    "axes.titleweight": "bold", "axes.titlesize": 13,
    "axes.labelsize": 11.5, "axes.labelweight": "medium",
    "legend.frameon": False, "legend.fontsize": 9.5,
    "xtick.labelsize": 10, "ytick.labelsize": 10,
    "figure.dpi": 150, "savefig.dpi": 300, "savefig.bbox": "tight",
    "savefig.facecolor": "white",
})
BAR_W = 0.5

def style_axis(ax, title=None, xlabel=None, ylabel=None):
    if title:  ax.set_title(title, fontweight="bold", pad=10)
    if xlabel: ax.set_xlabel(xlabel)
    if ylabel: ax.set_ylabel(ylabel)
    ax.grid(True, axis="y", linestyle="-", linewidth=0.7, color=GRID_COLOR, zorder=0)
    ax.set_axisbelow(True)
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)
    for spine in ("left", "bottom"):
        ax.spines[spine].set_color(AXIS_COLOR)
        ax.spines[spine].set_linewidth(1.0)
    return ax

def add_panel_label(ax, label):
    ax.text(-0.06, 1.08, label, transform=ax.transAxes, fontsize=14,
             fontweight="bold", va="top", ha="left", color=AXIS_COLOR)

def label_bars(ax, bars, values, fmt="{:.4f}", dy=0.015, fontsize=8.5):
    for b, v in zip(bars, values):
        ax.text(b.get_x() + b.get_width() / 2, v + dy, fmt.format(v), ha="center",
                 fontsize=fontsize, fontweight="bold", color=AXIS_COLOR,
                 path_effects=[pe.withStroke(linewidth=2.2, foreground="white")])

# ==============================================================================
# 1. SHARED TEXT PIPELINE — IDENTICAL normalization function used by every
# document-level fine-tuning script AND the Attention-Pooling script (the
# regex substitutions are the same everywhere in this project; only WHERE
# it's applied differs below — once per whole document for the six
# classifiers, once per sentence for Attention-Pooling, since that method
# needs per-sentence embeddings).
# ==============================================================================
_SENT_SPLIT_RE = re.compile(r'(?<=[.!?])\s+')
MIN_SENT_WORDS = 8

def sent_tokenize_en(text):
    if not isinstance(text, str):
        return []
    text = re.sub(r'\s*\n+\s*', ' ', text.strip())
    if not text:
        return []
    return [p.strip() for p in _SENT_SPLIT_RE.split(text) if p.strip()]

_INVISIBLE_CHARS_RE = re.compile('[​‌‍‎‏‪‫‬‭‮﻿]')
_CITATION_RE = re.compile(
    r'\[\d+(?:\s*,\s*\d+)*\]'
    r'|\([^()]{0,60}\d{4}[^()]{0,25}\)'
)
_URL_RE   = re.compile(r'(?:https?://|www\.)\S+')
_EMAIL_RE = re.compile(r'\b[\w.+-]+@[\w-]+\.[\w.-]+\b')
_EMOJI_RE = re.compile(
    "["
    "\U0001F300-\U0001FAFF"
    "\U00002600-\U000027BF"
    "\U0001F1E6-\U0001F1FF"
    "]+", flags=re.UNICODE
)
PUNCTUATION = r";:\"'`“”‘’«»\-–—/\\|@#\$%\^&\*_=~<>\(\)\[\]\{\}"
_NUMERIC_COMMA_SAFE_RE = re.compile(r'(?<!\d),|,(?!\d)')
_SPACE_BEFORE_SENT_END_RE = re.compile(r'\s+([.!?])')

_CONTRACTIONS = {
    "won't": "will not", "can't": "cannot", "n't": " not",
    "'re": " are", "'s": " is", "'d": " would", "'ll": " will",
    "'ve": " have", "'m": " am",
}
_CONTRACTIONS_RE = re.compile("|".join(re.escape(k) for k in _CONTRACTIONS.keys()))

def normalize_english_text(text):
    if not isinstance(text, str):
        return text
    text = _INVISIBLE_CHARS_RE.sub('', text)
    text = _CITATION_RE.sub(' ', text)
    text = _URL_RE.sub(' ', text)
    text = _EMAIL_RE.sub(' ', text)
    text = _EMOJI_RE.sub(' ', text)
    text = unicodedata.normalize('NFKC', text)
    text = _CONTRACTIONS_RE.sub(lambda m: _CONTRACTIONS[m.group(0)], text)
    text = _NUMERIC_COMMA_SAFE_RE.sub(" ", text)
    text = re.sub(rf"[{PUNCTUATION}]", " ", text)
    text = _SPACE_BEFORE_SENT_END_RE.sub(r'\1', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def normalize_english_text_attnpool(text):
    """Normalization used by the NEW AttnPool training script: identical to
    normalize_english_text but WITHOUT the final 'glue . ! ? to previous
    word' step (that step is not part of the training pipeline of this head)."""
    if not isinstance(text, str):
        return text
    text = _INVISIBLE_CHARS_RE.sub('', text)
    text = _CITATION_RE.sub(' ', text)
    text = _URL_RE.sub(' ', text)
    text = _EMAIL_RE.sub(' ', text)
    text = _EMOJI_RE.sub(' ', text)
    text = unicodedata.normalize('NFKC', text)
    text = _CONTRACTIONS_RE.sub(lambda m: _CONTRACTIONS[m.group(0)], text)
    text = _NUMERIC_COMMA_SAFE_RE.sub(" ", text)
    text = re.sub(rf"[{PUNCTUATION}]", " ", text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def doc_to_sentences_for_pooling(raw_text):
    """MODIFIED — mirrors the training script exactly: normalize the WHOLE
    document first, then split into sentences, keep those with >=
    MIN_SENT_WORDS words, and fall back to the whole normalized text if none
    qualify. The six classifiers do NOT use this."""
    norm = normalize_english_text_attnpool(raw_text)
    if not isinstance(norm, str) or not norm.strip():
        return []
    sents = [s for s in sent_tokenize_en(norm) if len(s.split()) >= MIN_SENT_WORDS]
    if not sents:
        sents = [norm]
    return sents

# ==============================================================================
# 2. CONFIG — عدّل المسارات هنا
# ==============================================================================
PROJECT_DIR = "/content/drive/MyDrive/english_ai_detector_project"

CLASSIFIER_PATHS = {
    "BERT":        f"{PROJECT_DIR}/BERT_DocumentLevel_FineTune/model_final_bert_doclevel",
    "RoBERTa":     f"{PROJECT_DIR}/RoBERTa_DocumentLevel_FineTune/model_final_roberta_doclevel",
    "ELECTRA":     f"{PROJECT_DIR}/ELECTRA_DocumentLevel_FineTune/model_final_electra_doclevel",
    "DeBERTa-v3":  f"{PROJECT_DIR}/DeBERTaV3_DocumentLevel_FineTune/model_final_deberta_v3_doclevel",
    "XLNet":       f"{PROJECT_DIR}/XLNet_DocumentLevel_FineTune/model_final_xlnet_doclevel",
    "DistilBERT":  f"{PROJECT_DIR}/DistilBERT_DocumentLevel_FineTune/model_final_distilbert_doclevel",
}

THRESHOLDS_DOC = {
    "BERT":        0.57,
    "RoBERTa":     0.55,
    "ELECTRA":     0.58,
    "DeBERTa-v3":  0.46,
    "XLNet":       0.52,
    "DistilBERT":  0.56,
    # ATTN_KEY's threshold is NOT hardcoded here — it is read directly from
    # its own saved head_config.json (th_doc) in Section 4 below, so this
    # script always uses whatever threshold that head's own validation-set
    # search actually found.
}

# Frozen SupCon (contrastive-learning) encoder — SAME checkpoint used by
# Dual-SVM — Attention-Pooling (Doc) was trained on top of this exact
# encoder (loaded, not retrained) by
# deberta_v3_supcon_attnpool_dochead_english.py.
M1_DRIVE_PATH = f"{PROJECT_DIR}/DeBERTaV3_SupCon_MLP_AttnPool"   # MODIFIED: new encoder + head folder
NEW_SUFFIX = "mlp_attnpool_deberta"                               # MODIFIED: suffix used by the training script
M1_ENCODER_NAME = "microsoft/deberta-v3-base"
M1_PROJ_HID, M1_PROJ_DIM = 512, 128
M1_MAX_LEN = 128   # per-sentence encoding length, same as that method's own scripts

# Attention-Pooling (Doc) head checkpoint, as saved by
# deberta_v3_supcon_attnpool_dochead_english.py.
ATTNPOOL_HEAD_DRIVE_PATH = M1_DRIVE_PATH   # MODIFIED: head is saved in the same folder as the encoder
ATTN_HIDDEN = 256       # must match that script's CFG["attn_hidden"]
HEAD_DROPOUT = 0.3      # unused at inference (eval mode), kept for class signature

# (see chat) — ONE merged external file: columns text, label (0=Human,
# 1=AI), model_used (generator name for AI rows, "Human-Written" for human
# rows). عدّل المسار حسب مكان رفع الملف على Colab.
EXTERNAL_DATA_FILE = "/content/Five.xlsx"
HUMAN_LABEL_VALUE = "Human-Written"

DOC_MAX_LEN = 512       # the six document-level classifiers — whole document as ONE sequence
DOC_INFER_BATCH = 16    # kept modest (vs. 32 at training) since MAX_LENGTH=512 sequences are long
OUTPUT_DIR = "./external_cross_generator_test_six_doclevel_plus_attnpool"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("=" * 70)
print("  External Cross-Generator Test — 6 Document-level classifiers + Proposed Method")
print("  DOCUMENT-LEVEL ONLY — no sentence-level reporting anywhere in this script")
print("=" * 70)
print(f"  Device: {DEVICE}")

# ==============================================================================
# 3. LOAD THE SIX DOCUMENT-LEVEL CLASSIFIERS
# ==============================================================================
print("\n" + "─" * 70)
print("  Loading the six DOCUMENT-LEVEL fine-tuned classifiers ...")
CLASSIFIERS = {}
for name, path in CLASSIFIER_PATHS.items():
    print(f"    {name:12s} <- {path}")
    tok = AutoTokenizer.from_pretrained(path)
    mdl = AutoModelForSequenceClassification.from_pretrained(path).to(DEVICE).eval()
    CLASSIFIERS[name] = (tok, mdl)
print("  All six document-level classifiers loaded.")

# ==============================================================================
# 4. LOAD Attention-Pooling (Doc) — frozen SupCon encoder (+ proj_head, kept
# only so the saved state_dict's keys match; never used at inference) +
# the trained AttnPoolDocHead. No sentence SVM / document SVM anywhere in
# this script.
# ==============================================================================
print("\n" + "─" * 70)
print("  Loading Proposed Method checkpoint ...")

class ProjectionHead(nn.Module):
    def __init__(self, in_dim, hid, out):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_dim, hid), nn.BatchNorm1d(hid),
                                 nn.ReLU(inplace=True), nn.Linear(hid, out))
    def forward(self, x): return F.normalize(self.net(x), dim=-1)

class MLPHead(nn.Module):
    """Same class as in the training script: Linear(in->in//2) -> ReLU ->
    Dropout -> Linear(in//2 -> num_labels)."""
    def __init__(self, in_dim, num_labels=2, dropout_prob=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, in_dim // 2),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout_prob),
            nn.Linear(in_dim // 2, num_labels),
        )
    def forward(self, x):
        return self.net(x)

class SharedModel(nn.Module):
    def __init__(self, name, proj_hid, proj_dim, num_labels=2, cls_dropout=0.3):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(name, torch_dtype=torch.float32)
        hs = self.encoder.config.hidden_size
        self.proj_head = ProjectionHead(hs, proj_hid, proj_dim)
        self.cls_head = MLPHead(hs, num_labels=num_labels, dropout_prob=cls_dropout)   # sentence head (kept so strict load matches; unused here)
    def get_pooled(self, ids, mask):
        hidden = self.encoder(input_ids=ids, attention_mask=mask).last_hidden_state
        mask_expanded = mask.unsqueeze(-1).float()
        summed = (hidden * mask_expanded).sum(dim=1)
        counts = mask_expanded.sum(dim=1).clamp(min=1e-9)
        return summed / counts

class AttnPoolDocHead(nn.Module):
    """IDENTICAL architecture to
    deberta_v3_supcon_attnpool_dochead_english.py's AttnPoolDocHead: a
    small scoring network gives each sentence a learned weight (masked
    softmax over the document's own sentences); the document vector is
    their WEIGHTED SUM -> small MLP classifier -> 1 logit."""
    def __init__(self, in_dim, attn_hidden=256, dropout=0.3):
        super().__init__()
        self.attn = nn.Sequential(
            nn.Linear(in_dim, attn_hidden), nn.Tanh(), nn.Linear(attn_hidden, 1),
        )
        self.classifier = MLPHead(in_dim, num_labels=1, dropout_prob=dropout)   # MODIFIED: same MLPHead class as training
    def forward(self, x, mask):
        scores = self.attn(x).squeeze(-1)
        scores = scores.masked_fill(~mask, float("-inf"))
        weights = torch.softmax(scores, dim=1)
        weights = torch.nan_to_num(weights, nan=0.0)
        pooled = (x * weights.unsqueeze(-1)).sum(dim=1)
        logit = self.classifier(pooled).squeeze(-1)
        return logit, pooled, weights

m1_tokenizer = AutoTokenizer.from_pretrained(f"{M1_DRIVE_PATH}/tokenizer_{NEW_SUFFIX}")
m1_model = SharedModel(M1_ENCODER_NAME, M1_PROJ_HID, M1_PROJ_DIM).to(DEVICE)
m1_ckpt = torch.load(f"{M1_DRIVE_PATH}/encoder_{NEW_SUFFIX}.pt", map_location=DEVICE, weights_only=False)
m1_model.load_state_dict(m1_ckpt["shared_model"], strict=True)
m1_model = m1_model.float().to(DEVICE).eval()
for p in m1_model.parameters():
    p.requires_grad = False
M1_HIDDEN = m1_model.encoder.config.hidden_size

attn_doc_head = AttnPoolDocHead(M1_HIDDEN, ATTN_HIDDEN, HEAD_DROPOUT).to(DEVICE)
_attn_ckpt = torch.load(f"{ATTNPOOL_HEAD_DRIVE_PATH}/attnpool_doc_head_{NEW_SUFFIX}.pt", map_location=DEVICE, weights_only=False)
attn_doc_head.load_state_dict(_attn_ckpt["attn_doc_head_state_dict"])
attn_doc_head = attn_doc_head.float().to(DEVICE).eval()
for p in attn_doc_head.parameters():
    p.requires_grad = False

with open(f"{ATTNPOOL_HEAD_DRIVE_PATH}/config_{NEW_SUFFIX}.json") as f:
    _attn_head_cfg = json.load(f)
THRESHOLDS_DOC[ATTN_KEY] = float(_attn_head_cfg["th_doc"])

print(f"  Proposed Method loaded. Encoder={M1_ENCODER_NAME}  hidden={M1_HIDDEN}  "
      f"threshold={THRESHOLDS_DOC[ATTN_KEY]:.2f}  "
      f"val F1={_attn_head_cfg.get('s3_best_val_f1', float('nan')):.4f}")

print(f"  Thresholds in use: {', '.join(f'{m}={THRESHOLDS_DOC[m]:.2f}' for m in METHOD_ORDER)}")

# ==============================================================================
# 5. INFERENCE — one function per method family, BOTH document-level only
# ==============================================================================
@torch.no_grad()
def predict_probs_doclevel_classifier(method_name, texts, batch_size=DOC_INFER_BATCH):
    """Whole DOCUMENT as ONE sequence (MAX_LENGTH=512), straight to
    softmax — exactly like that backbone's own
    *_finetune_document_level_english.py training/testing scripts."""
    tok, mdl = CLASSIFIERS[method_name]
    probs = []
    n_batches = (len(texts) + batch_size - 1) // batch_size
    for i in tqdm(range(0, len(texts), batch_size), total=n_batches,
                  desc=f"  {method_name} (document-level)"):
        batch = texts[i:i + batch_size]
        enc = tok(batch, truncation=True, max_length=DOC_MAX_LEN, padding=True,
                  return_tensors="pt").to(DEVICE)
        logits = mdl(**enc).logits
        probs.append(torch.softmax(logits, dim=-1)[:, 1].cpu().numpy())
    return np.concatenate(probs) if probs else np.array([])

@torch.no_grad()
def encode_sentences_m1(sent_list, batch_size=32):
    if not sent_list:
        return np.zeros((0, M1_HIDDEN), dtype=np.float32)
    embs = []
    for i in range(0, len(sent_list), batch_size):
        batch = sent_list[i:i + batch_size]
        enc = m1_tokenizer(batch, truncation=True, max_length=M1_MAX_LEN, padding=True,
                           return_tensors="pt").to(DEVICE)
        pooled = m1_model.get_pooled(enc["input_ids"], enc["attention_mask"])
        embs.append(pooled.cpu().numpy().astype(np.float32))
    return np.concatenate(embs, axis=0)

@torch.no_grad()
def predict_prob_attnpool_doc(raw_text):
    """Attention-Pooling (Doc) — frozen SupCon sentence embeddings, pooled
    by the trained head's learned per-sentence attention weights (weighted
    sum) instead of an unweighted mean."""
    sents = doc_to_sentences_for_pooling(raw_text)
    if not sents:
        return 0.5   # no usable sentence found -> neutral, no-evidence baseline
    sent_emb = encode_sentences_m1(sents)
    x = torch.from_numpy(sent_emb).unsqueeze(0).to(DEVICE)
    mask = torch.ones(1, sent_emb.shape[0], dtype=torch.bool, device=DEVICE)
    logit, _, _ = attn_doc_head(x, mask)
    return float(torch.sigmoid(logit).item())

# ==============================================================================
# 6. EVALUATE THE WHOLE MERGED FILE — DOCUMENT level, ALL methods
# ==============================================================================
def evaluate_all(fdf):
    texts_raw = fdf["text"].tolist()
    labels = fdf["label"].astype(int).tolist()
    gens = fdf["model_used"].astype(str).tolist()

    texts_norm_doc = [normalize_english_text(t) for t in texts_raw]
    doc_probs = {}
    for m in CLASSIFIER_PATHS:
        doc_probs[m] = predict_probs_doclevel_classifier(m, texts_norm_doc)   # own tqdm bar inside

    attn_probs = []
    for t in tqdm(texts_raw, desc=f"  Scoring with {ATTN_KEY}"):
        attn_probs.append(predict_prob_attnpool_doc(t))
    doc_probs[ATTN_KEY] = np.array(attn_probs)

    doc_df = pd.DataFrame({"generator": gens, "doc_idx": range(len(fdf)), "label": labels})
    for m in METHOD_ORDER:
        doc_df[f"p_{m}"] = doc_probs[m]
    return doc_df

def compute_metrics(probs, labels, threshold):
    preds = (probs >= threshold).astype(int)
    acc = accuracy_score(labels, preds)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="binary", zero_division=0)
    mcc = matthews_corrcoef(labels, preds)
    fpr, tpr, _ = roc_curve(labels, probs)
    roc_auc = auc(fpr, tpr)
    return {"N": len(probs), "Threshold": round(float(threshold), 2),
            "Accuracy": round(acc, 4), "F1": round(f1, 4), "Precision": round(p, 4),
            "Recall": round(r, 4), "MCC": round(mcc, 4), "AUC": round(roc_auc, 4)}

def breakdown_by_generator_and_human(df_lvl, thresholds):
    """One row of Correct_rate per unseen AI generator (recall, on that
    generator's label==1 rows only), PLUS one "Human" row (specificity, on
    ALL label==0 rows)."""
    y_true = df_lvl["label"].values
    gen_arr = df_lvl["generator"].astype(str).values
    rows = []

    ai_generators = sorted(set(gen_arr[y_true == 1]) - {HUMAN_LABEL_VALUE})
    for g in ai_generators:
        m = (gen_arr == g) & (y_true == 1)
        if m.sum() == 0:
            continue
        for method in METHOD_ORDER:
            preds = (df_lvl[f"p_{method}"].values[m] >= thresholds[method]).astype(int)
            correct_rate = float((preds == 1).mean())
            rows.append({"generator": g, "Method": method, "n": int(m.sum()),
                        "Correct_rate": round(correct_rate, 4)})

    human_mask = y_true == 0
    for method in METHOD_ORDER:
        preds = (df_lvl[f"p_{method}"].values[human_mask] >= thresholds[method]).astype(int)
        correct_rate = float((preds == 0).mean())
        rows.append({"generator": "Human", "Method": method, "n": int(human_mask.sum()),
                    "Correct_rate": round(correct_rate, 4)})

    return pd.DataFrame(rows)

# ==============================================================================
# 7. LOAD THE MERGED FILE AND RUN THE TEST ONCE
# ==============================================================================
print("\n" + "=" * 70)
print(f"  Loading merged external file: {EXTERNAL_DATA_FILE}")
print("=" * 70)

fdf = pd.read_excel(EXTERNAL_DATA_FILE)
if "content" in fdf.columns and "text" not in fdf.columns:
    fdf = fdf.rename(columns={"content": "text"})
required_cols = {"text", "label", "model_used"}
missing = required_cols - set(fdf.columns)
if missing:
    raise ValueError(f"Missing required column(s) {missing} in {EXTERNAL_DATA_FILE}. "
                      f"Found columns: {list(fdf.columns)}")

fdf["text"] = fdf["text"].astype(str)
fdf = fdf[fdf["text"].str.len() > 20].reset_index(drop=True)
fdf["label"] = fdf["label"].astype(int)
fdf["model_used"] = fdf["model_used"].astype(str)

print(f"  Total texts: {len(fdf)}")
print(f"\n  label distribution:\n{fdf['label'].value_counts().to_string()}")
print(f"\n  model_used distribution:\n{fdf['model_used'].value_counts().to_string()}")

doc_df = evaluate_all(fdf)
doc_df.to_csv(os.path.join(OUTPUT_DIR, "document_level_predictions.csv"), index=False)

# ── نتائج عامة (كل البيانات مع بعض)، مستوى المستند فقط ──
y_true = doc_df["label"].values
overall_rows = []
for m in METHOD_ORDER:
    row = compute_metrics(doc_df[f"p_{m}"].values, y_true, THRESHOLDS_DOC[m])
    row.update({"Method": m, "Level": "Document"})
    overall_rows.append(row)

overall_df = pd.DataFrame(overall_rows)
print("\n" + "=" * 70)
print("  OVERALL RESULTS — all generators + Human combined, document-level, ALL 7 METHODS")
print("=" * 70)
print(overall_df.to_string(index=False))
overall_df.to_csv(os.path.join(OUTPUT_DIR, "overall_results.csv"), index=False)

# ── مصفوفة الارتباك (مستوى المستند فقط) ──
fig1, axes1 = plt.subplots(1, len(METHOD_ORDER), figsize=(4.0 * len(METHOD_ORDER), 5.4))
fig1.suptitle("Confusion Matrix — Document-level (all generators + Human)",
              fontsize=16, fontweight="bold", y=1.04)
for i, (m, cmp_) in enumerate(zip(METHOD_ORDER, CM_CMAPS)):
    ax = axes1[i]
    preds = (doc_df[f"p_{m}"].values >= THRESHOLDS_DOC[m]).astype(int)
    cm = confusion_matrix(y_true, preds)
    sns.heatmap(cm, annot=True, fmt="d", cmap=cmp_, ax=ax,
                xticklabels=["Human", "AI"], yticklabels=["Human", "AI"],
                annot_kws={"size": 13, "weight": "bold"}, cbar=False,
                linewidths=0, square=True)
    tn_, fp_, fn_, tp_ = cm.ravel()
    f1v = overall_df[overall_df["Method"] == m]["F1"].values[0]
    accv = overall_df[overall_df["Method"] == m]["Accuracy"].values[0]
    ax.set_title(f"{disp(m)}\n(th={THRESHOLDS_DOC[m]:.2f})  F1={f1v:.4f}  Acc={accv:.4f}",
                 fontweight="bold", fontsize=9.5, pad=9)
    ax.set_ylabel("Actual", fontsize=9.5)
    ax.set_xlabel(f"Predicted\nTN={tn_} FP={fp_} FN={fn_} TP={tp_}", fontsize=7.8)
    add_panel_label(ax, chr(65 + i))
plt.tight_layout()
p1 = os.path.join(OUTPUT_DIR, "confusion_matrices_document.png")
plt.savefig(p1, bbox_inches="tight"); plt.show()

# ── ROC (مستوى المستند فقط — لوحة واحدة) ──
fig2, ax2 = plt.subplots(figsize=(7.5, 6.2))
for m, color in zip(METHOD_ORDER, METHOD_COLORS):
    fpr, tpr, _ = roc_curve(y_true, doc_df[f"p_{m}"].values)
    a = auc(fpr, tpr)
    ax2.plot(fpr, tpr, color=color, lw=2.4, label=f"{disp(m)} (AUC={a:.4f})")
ax2.plot([0, 1], [0, 1], "--", color="#9AA0AC", lw=1.2, label="Chance (AUC=0.5)")
style_axis(ax2, "Document-level ROC — all generators + Human", "False Positive Rate", "True Positive Rate")
ax2.legend(fontsize=8.5, loc="lower right")
plt.tight_layout()
p2 = os.path.join(OUTPUT_DIR, "roc_curve_document.png")
plt.savefig(p2, bbox_inches="tight"); plt.show()

# ── مقارنة F1 (مستوى المستند فقط — عمود واحد لكل طريقة) ──
fig3, ax3 = plt.subplots(figsize=(10, 5.6))
x = np.arange(len(METHOD_ORDER))
doc_f1 = overall_df["F1"].values
b1 = ax3.bar(x, doc_f1, width=BAR_W, color=METHOD_COLORS, edgecolor="white", linewidth=1.2, zorder=3)
ax3.set_xticks(x); ax3.set_xticklabels([disp(m) for m in METHOD_ORDER], rotation=15)
style_axis(ax3, "F1 — Document-level (all generators + Human)", ylabel="F1")
ax3.set_ylim(0, 1.08)
label_bars(ax3, b1, doc_f1)
plt.tight_layout()
p3 = os.path.join(OUTPUT_DIR, "f1_document.png")
plt.savefig(p3, bbox_inches="tight"); plt.show()

# ==============================================================================
# 8. CENTERPIECE — Human accuracy (once) + each generator's accuracy (once),
# per method, DOCUMENT level only.
# ==============================================================================
print("\n" + "=" * 70)
print("  BREAKDOWN — Human (specificity) + each unseen generator (recall), document-level, all 7 methods")
print("=" * 70)

breakdown_df = breakdown_by_generator_and_human(doc_df, THRESHOLDS_DOC)
breakdown_df.to_csv(os.path.join(OUTPUT_DIR, "breakdown_by_generator_document.csv"), index=False)

pivot = breakdown_df.pivot(index="generator", columns="Method", values="Correct_rate")[METHOD_ORDER]
pivot = pivot.loc[["Human"] + sorted(set(pivot.index) - {"Human"})]

pivot.loc["Mean"] = pivot.mean(axis=0)

print(f"\n  Document-level:")
print(pivot.round(4).to_string())

pivot_disp = pivot.rename(columns=disp)
n_rows = len(pivot)
fig4, ax4 = plt.subplots(figsize=(2.2 * len(METHOD_ORDER), max(4, 0.65 * n_rows + 2)))
pivot_f = pivot_disp.astype(float)
sns.heatmap(
    pivot_f, annot=False, cmap=BLUE_ONLY_CMAP, vmin=0, vmax=1,
    linewidths=0.8, linecolor="white",
    cbar_kws={"label": "Correct rate (recall on AI / specificity on Human)", "shrink": 0.85},
    ax=ax4, square=False,
)
heatmap_dynamic_text(ax4, pivot_f, 0, 1, BLUE_ONLY_CMAP, fontsize=11)
ax4.axhline(1, color=AXIS_COLOR, linewidth=2.2)   # separates Human from the AI generators
ax4.axhline(n_rows - 1, color=AXIS_COLOR, linewidth=2.2)   # separates the AI generators from the final "Mean" row
ax4.set_title(f"Detection Rate by AI Generator + Human — Document-level — All {len(METHOD_ORDER)} Methods",
             fontsize=14, fontweight="bold", pad=12)
ax4.set_xlabel(""); ax4.set_ylabel("")
ax4.tick_params(axis="y", rotation=0)
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
hpath = os.path.join(OUTPUT_DIR, "heatmap_by_generator_document.png")
plt.savefig(hpath, bbox_inches="tight"); plt.show()
print(f"  Saved -> {hpath}")

# ==============================================================================
# 8b. MISCLASSIFIED DOCUMENTS — now centered on Attention-Pooling (Doc)
# instead of Dual-SVM (see chat, 2026-10-20), same export structure as
# before: all misclassified docs, an error-rate-by-generator summary, and
# the subset ONLY this method got wrong while every other method got right.
# ==============================================================================
print("\n" + "─" * 70)
print("  Extracting Proposed Method misclassified documents ...")

attn_probs = doc_df[f"p_{ATTN_KEY}"].values
attn_preds = (attn_probs >= THRESHOLDS_DOC[ATTN_KEY]).astype(int)
attn_wrong_mask = attn_preds != y_true   # fdf and doc_df share the same row order/index

attn_errors_df = fdf.loc[attn_wrong_mask, ["text", "model_used"]].copy()
attn_errors_df.insert(0, "doc_idx", doc_df.loc[attn_wrong_mask, "doc_idx"].values)
attn_errors_df = attn_errors_df.rename(columns={"model_used": "generator_model"})
attn_errors_df["true_label"] = y_true[attn_wrong_mask]
attn_errors_df["true_label_name"] = np.where(attn_errors_df["true_label"] == 1, "AI", "Human")
attn_errors_df["predicted_label"] = attn_preds[attn_wrong_mask]
attn_errors_df["predicted_label_name"] = np.where(attn_errors_df["predicted_label"] == 1, "AI", "Human")
attn_errors_df["prob_ai"] = attn_probs[attn_wrong_mask]
attn_errors_df["threshold_used"] = THRESHOLDS_DOC[ATTN_KEY]
attn_errors_df["error_type"] = np.where(
    (attn_errors_df["true_label"] == 0) & (attn_errors_df["predicted_label"] == 1),
    "False Positive (Human -> AI)",
    np.where(
        (attn_errors_df["true_label"] == 1) & (attn_errors_df["predicted_label"] == 0),
        "False Negative (AI -> Human)", "",
    ),
)
attn_errors_df = attn_errors_df[[
    "doc_idx", "generator_model", "text", "true_label_name", "predicted_label_name",
    "error_type", "prob_ai", "threshold_used", "true_label", "predicted_label",
]].sort_values("doc_idx").reset_index(drop=True)

error_summary_rows = []
for gen in sorted(fdf["model_used"].unique()):
    total_n = int((fdf["model_used"] == gen).sum())
    wrong_n = int((attn_errors_df["generator_model"] == gen).sum())
    error_summary_rows.append({
        "generator_model": gen,
        "total_documents": total_n,
        "misclassified_documents": wrong_n,
        "error_rate": round(wrong_n / total_n, 4) if total_n else 0.0,
    })
error_summary_df = pd.DataFrame(error_summary_rows).sort_values(
    "misclassified_documents", ascending=False).reset_index(drop=True)

preds_by_method = {m: (doc_df[f"p_{m}"].values >= THRESHOLDS_DOC[m]).astype(int) for m in METHOD_ORDER}
correct_by_method = {m: (preds_by_method[m] == y_true) for m in METHOD_ORDER}
other_methods = [m for m in METHOD_ORDER if m != ATTN_KEY]
all_others_correct = np.all([correct_by_method[m] for m in other_methods], axis=0)
attn_unique_wrong_mask = (~correct_by_method[ATTN_KEY]) & all_others_correct

attn_unique_errors_df = fdf.loc[attn_unique_wrong_mask, ["text", "model_used"]].copy()
attn_unique_errors_df.insert(0, "doc_idx", doc_df.loc[attn_unique_wrong_mask, "doc_idx"].values)
attn_unique_errors_df = attn_unique_errors_df.rename(columns={"model_used": "generator_model"})
attn_unique_errors_df["true_label_name"] = np.where(y_true[attn_unique_wrong_mask] == 1, "AI", "Human")
attn_unique_errors_df["attnpool_predicted_label_name"] = np.where(
    preds_by_method[ATTN_KEY][attn_unique_wrong_mask] == 1, "AI", "Human")
attn_unique_errors_df["attnpool_prob_ai"] = attn_probs[attn_unique_wrong_mask]
for m in other_methods:
    attn_unique_errors_df[f"p_{m}"] = doc_df.loc[attn_unique_wrong_mask, f"p_{m}"].values
attn_unique_errors_df = attn_unique_errors_df.sort_values("doc_idx").reset_index(drop=True)

attn_errors_path = os.path.join(OUTPUT_DIR, "attnpool_misclassified_documents.xlsx")
with pd.ExcelWriter(attn_errors_path, engine="openpyxl") as writer:
    attn_errors_df.to_excel(writer, sheet_name="Misclassified", index=False)
    error_summary_df.to_excel(writer, sheet_name="Error_Summary_by_Generator", index=False)
    attn_unique_errors_df.to_excel(writer, sheet_name="Unique_Attn_Errors", index=False)

print(f"  Proposed Method got {len(attn_errors_df)}/{len(fdf)} documents wrong "
      f"({len(attn_errors_df) / len(fdf):.1%}) -> saved to {attn_errors_path} "
      f"(sheets: Misclassified, Error_Summary_by_Generator, Unique_Attn_Errors)")
print(f"\n  Error summary by generator model:\n{error_summary_df.to_string(index=False)}")
print(f"\n  Documents ONLY Proposed Method got wrong (all six classifiers correct): "
      f"{len(attn_unique_errors_df)}/{len(fdf)} ({len(attn_unique_errors_df) / len(fdf):.1%})")

# ==============================================================================
# 9. SAVE EVERYTHING TO DRIVE
# ==============================================================================
print("\n" + "─" * 70)
print("  Saving all external test results to Drive ...")

drive_out = os.path.join(PROJECT_DIR, "external_cross_generator_test_six_doclevel_plus_attnpool")
os.makedirs(drive_out, exist_ok=True)
for fn in os.listdir(OUTPUT_DIR):
    fp = os.path.join(OUTPUT_DIR, fn)
    if os.path.isfile(fp):
        shutil.copy(fp, drive_out)

print(f"  Saved -> {drive_out}")

try:
    from google.colab import files
    files.download(attn_errors_path)
except Exception as e:
    print(f"  (could not trigger an automatic browser download here: {e}. "
          f"The file is still saved at {attn_errors_path} and in Drive at "
          f"{os.path.join(drive_out, 'attnpool_misclassified_documents.xlsx')}.)")

print("\n" + "=" * 70)
print("  EXTERNAL CROSS-GENERATOR TEST COMPLETE")
print("  (6 document-level classifiers + Proposed Method, document-level only)")
print("=" * 70)